# Load an entity: TransactionHistory (local)

Loads `TransactionHistory` into its SCD target on a local Spark session, from a Parquet file through a DQD, or through two staging notebooks (`IS_DQD`).
Everything the load needs is one JSON load context; every section and field is described in
[docs/reference/load-context.md](../../docs/reference/load-context.md).

The notebook runs top to bottom:

1. **Input parameters**: the run switches and the load context.
2. **Library**: `spark_quick_start` from this repository's `src/`, with no install.
3. **Imports**.
4. **Spark session**: local, with Delta.
5. **Pipeline**: built from the switches and run; the result is the cell's output.
6. **Change one row and load again**: a second load sees one changed row and keeps both versions.

It expects a kernel with PySpark and `delta-spark`, started anywhere inside the repository.
The same notebook runs on macOS, Linux and Windows; on Windows it also needs `HADOOP_HOME`
pointing at a Hadoop folder whose `bin/` holds `winutils.exe` and `hadoop.dll`.

## 1. Input parameters

In [ ]:
from pathlib import Path

# The repository root: the nearest folder at or above the working directory holding pyproject.toml.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())


def repo_path(*parts: str) -> str:
    """A path inside the repository, as the load context expects it."""
    return REPO_ROOT.joinpath(*parts).as_posix()


def struct(*fields: tuple[str, str]) -> dict:
    """A Spark StructType as JSON, every field nullable."""
    return {"type": "struct",
            "fields": [{"name": n, "type": t, "nullable": True, "metadata": {}} for n, t in fields]}

In [ ]:
# Run switches
IS_DQD = True             # True: the source is dqd_context; False: the staging notebooks
APPEND_ONLY = False       # True: NativeSCDAppendOnlyTransformer; False: NativeSCDMergeTransformer (SCD2)
DRIVER_MEMORY = "1g"      # Spark's default; raise it for a large load (backlog F153)
SHUFFLE_PARTITIONS = 200

SOURCE_PARQUET = repo_path("data", "Production_TransactionHistory",
                           "part-00000-f8e129a4-417b-4815-bb81-9c4e9b4b0770-c000.snappy.parquet")
STAGING_NOTEBOOKS = repo_path("reference_design", "local", "StagingNB")
SOURCE_COLUMNS = ["TransactionID", "ProductID", "ReferenceOrderID", "ReferenceOrderLineID",
                  "TransactionDate", "TransactionType", "Quantity", "ActualCost", "ModifiedDate"]

# The load context: the five required sections, plus the optional hwm_context (the watermark)
# and dqd_context (a Declarative Query Definition source). With both staging stages and a
# dqd_context present, the load logs a warning naming the one IS_DQD leaves unused.
load_context_dict = {
    "global_context": {
        "source_system": "CS",
        "id": "TransactionHistory",
        "entity_name": "TransactionHistory",
        "staging_table_schema": struct(
            ("TransactionID", "integer"), ("ProductID", "integer"), ("ReferenceOrderID", "integer"),
            ("ReferenceOrderLineID", "integer"), ("TransactionDate", "timestamp"),
            ("TransactionType", "string"), ("Quantity", "integer"), ("ActualCost", "decimal(19,4)"),
            ("ModifiedDate", "timestamp"), ("scdDeletedFlag", "string"), ("scdRowKey", "string"),
        ),
        "staging_table_path": repo_path("data", "Staging", "Production_TransactionHistory"),
        "target_table_path": repo_path("data", "Target", "Production_TransactionHistory"),
    },
    "staging_context": {
        "stages": [
            {"nb_name": "NB_BI_Stage_Gen3_TransactionHistory_A", "path": STAGING_NOTEBOOKS,
             "staging_context": {"src_parquet_path": SOURCE_PARQUET}},
            {"nb_name": "NB_BI_Stage_Gen3_TransactionHistory_B", "path": STAGING_NOTEBOOKS,
             "staging_context": {"src_parquet_path": SOURCE_PARQUET}},
        ]
    },
    "scd_context": {
        "business_key_column": "scdRowKey",
        "row_key_column": "scdRowKey",
        "source_deleted_flag_column": "scdDeletedFlag",
        "correlation_id_column": "scdCorrelationId",
        "current_flag_value": "Y",
        "default_max_timestamp": "2999-12-31 23:59:59.999999",
        "deleted_flag_column": "scdDeletedFlag",
        "deleted_flag_value": "Y",
        "effective_from_column": "scdEffectiveFrom",
        "effective_to_column": "scdEffectiveTo",
        "hash_column": "scdHash",
        "hash_separator": "~",
        "inactive_flag_value": "N",
        "load_type": "Incremental",
        "not_deleted_flag_value": "N",
        "scd_type": "scd2",
        "source_system_column": "scdSourceSystem",
        "version_column": "scdVersion",
        "exclude_from_hash": ["Quantity"],
    },
    "runtime_context": {
        "application_name": "SilverLoad",
        "max_parallel_staging": 5,
        "correlation_id": "local-demo",
        "pipeline_name": "reference-design-local",
    },
    "logging_context": {
        "log_file_dir": repo_path("data", "Logs"),
        "logging_level": "INFO",
        "log_file_name": "Bronze_TransactionHistory",
        "verbose": False,
    },
    "hwm_context": {
        "hwm_column_name": "ModifiedDate",
        "hwm_value": "2014-08-01 00:00:00",
    },
    "dqd_context": {
        "source_kind": "parquet",
        "source_identifier": SOURCE_PARQUET,
        "source_columns": SOURCE_COLUMNS,
        "transformations": [
            {"name": "TransactionID_KeyColumn_str",
             "expression": {"function": "cast", "arguments": [{"column": "TransactionID"}, {"literal": "string"}]}},
            {"name": "TransactionID_KeyColumn_str_notnull",
             "expression": {"function": "coalesce", "arguments": [{"column": "TransactionID_KeyColumn_str"}, {"literal": ""}]}},
            {"name": "scdRowKey",
             "expression": {"function": "concat_ws", "arguments": [
                 {"literal": "~"}, {"literal": "AW2022"}, {"column": "TransactionID_KeyColumn_str_notnull"}]}},
        ],
        "selection": [
            *({"name": c} for c in SOURCE_COLUMNS),
            {"name": "scdDeletedFlag", "expression": {"literal": "N"}},
            {"name": "scdRowKey"},
        ],
    },
}

## 2. Library

In [ ]:
import sys

# The library as a source copy (./src on sys.path), the way it also ships: nothing to install.
# Put first, so an installed copy cannot shadow it. Re-running this cell and the imports cell
# picks up edits to src/ without restarting the kernel.
src = str(REPO_ROOT / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for name in [m for m in sys.modules if m == "spark_quick_start" or m.startswith("spark_quick_start.")]:
    del sys.modules[name]

import spark_quick_start

print(f"spark_quick_start from {Path(spark_quick_start.__file__).parent}")

## 3. Imports

In [ ]:
import json
import os
import platform
from importlib.metadata import version

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from spark_quick_start.adapters import LocalAdapter
from spark_quick_start.orchestration import LoadOrchestrator
from spark_quick_start.orchestration.transformers import (
    CandidateGuardTransformer,
    SCDChangeDetectorTransformer,
    DQDTransformer,
    MetricsChartTransformer,
    NativeSCDAppendOnlyTransformer,
    NativeSCDMergeTransformer,
    ProcessTransformer,
    StagingTransformer,
    SCDTargetTableTransformer,
)

## 4. Spark session

In [ ]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(load_context_dict["runtime_context"]["application_name"])
    .master("local[*]")
    # Local mode runs the driver and every task in one JVM, sharing this heap. It is fixed when
    # the JVM starts: a new value needs a kernel restart, not only a new session.
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

if os.name == "nt":
    # Windows only: Spark reaches the local file system through Hadoop's winutils.exe and
    # hadoop.dll, found in HADOOP_HOME/bin, and must not try Hadoop's native Unix libraries.
    hadoop_home = os.environ.get("HADOOP_HOME")
    if not hadoop_home:
        raise EnvironmentError("Set HADOOP_HOME to a folder whose bin/ holds winutils.exe and hadoop.dll.")
    hadoop_bin = os.path.join(hadoop_home, "bin")
    if hadoop_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")
    builder = (
        builder
        .config("spark.hadoop.io.native.lib.available", "false")
        .config("spark.driver.extraLibraryPath", hadoop_bin)
        .config("spark.executor.extraLibraryPath", hadoop_bin)
    )

# delta-spark adds the Delta package that matches its own version and this Spark's Scala build
# (delta-spark_2.12 on Spark 3.5, delta-spark_4.x_2.13 on Spark 4), from the Ivy cache when present.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")

print(f"Spark {spark.version} on {platform.system()}, delta-spark {version('delta-spark')}, "
      f"driver memory {spark.sparkContext.getConf().get('spark.driver.memory', '1g')}")

## 5. Pipeline

### Optional: steps for trying out changes

Not part of the pipeline below. Add one after the source, for example
`.add(PatchTransformer("scdRowKey", "AW2022~213376", "ActualCost", 99.99))`, to see a change
detected, as section 6 does. A change to `Quantity` makes no new version: the load context lists it
in `exclude_from_hash`.
Setting `spark.driver.memory` only takes effect on a new session: restart the kernel after
changing `DRIVER_MEMORY`.

In [ ]:
class MarkRowAsDeleted(ProcessTransformer):
    """Flag one source row as deleted before change detection."""

    def __init__(self, key_column: str, key_value) -> None:
        self._key_column = key_column
        self._key_value = key_value

    def _run(self, state, ctx):
        flag = ctx.scd_ctx.deleted_flag_column
        return state.with_df(state.df.withColumn(
            flag, F.when(F.col(self._key_column) == F.lit(self._key_value), F.lit("Y")).otherwise(F.col(flag))))


class PatchTransformer(ProcessTransformer):
    """Overwrite one cell of the source row whose key_column equals key_value."""

    def __init__(self, key_column: str, key_value, column_name: str, new_value) -> None:
        self._key_column, self._key_value = key_column, key_value
        self._column_name, self._new_value = column_name, new_value

    def _run(self, state, ctx):
        return state.with_df(state.df.withColumn(
            self._column_name,
            F.when(F.col(self._key_column) == F.lit(self._key_value),
                   F.lit(self._new_value).cast(state.df.schema[self._column_name].dataType))
            .otherwise(F.col(self._column_name))))

In [ ]:
adapter = LocalAdapter(base_path=repo_path(), notebooks_dir=repo_path("reference_design", "local"))



def load(*extra_steps):
    """Run the load once; any extra steps go right after the source."""
    pipeline = (
        LoadOrchestrator(json.dumps(load_context_dict), spark, adapter=adapter)
        .set_source(IS_DQD, DQDTransformer(), StagingTransformer())
    )
    for step in extra_steps:
        pipeline = pipeline.add(step)
    return (
        pipeline
        .add(SCDTargetTableTransformer())
        .add(SCDChangeDetectorTransformer())
        .add(CandidateGuardTransformer())
        .add_if(APPEND_ONLY, NativeSCDAppendOnlyTransformer())
        .add_if(not APPEND_ONLY, NativeSCDMergeTransformer())
        .add(MetricsChartTransformer())
    ).run()


result = load()
print(result)

### Inspect the target: its latest current rows

In [ ]:
target: DataFrame = spark.read.format("delta").load(load_context_dict["global_context"]["target_table_path"])
target.filter(F.col("scdCurrentFlag") == "Y").orderBy(F.col("scdEffectiveFrom").desc()).show(5, truncate=False)

## 6. Change one row and load again

The second load patches one source row's `ActualCost` on its way in, so the change detector sees
exactly one changed row. With the SCD2 merge (`APPEND_ONLY = False`), the target closes the old
version and adds a new one: the result reports one row updated, and the key's history shows two
versions, only the newest current. Running the whole notebook again keeps going: section 5
compares the unpatched source with the patched target and adds version 3, and section 6 adds
version 4. Delete `data/Target/` and `data/Staging/` to start over.

In [ ]:
KEY = "AW2022~213376"

second = load(PatchTransformer("scdRowKey", KEY, "ActualCost", 99.99))
print(second)

history = (spark.read.format("delta").load(load_context_dict["global_context"]["target_table_path"])
           .filter(F.col("scdRowKey") == KEY)
           .orderBy("scdVersion"))
history.select("scdRowKey", "ActualCost", "scdVersion", "scdCurrentFlag",
               "scdEffectiveFrom", "scdEffectiveTo").show(truncate=False)